# Module 05 — Pre-election monitoring dashboard
Run in Colab after Modules 01–04. This dashboard reads saved CSVs only: no scraping, API calls or scoring. Refresh reloads outputs after you rerun the upstream notebooks. It does not automatically collect new data.

Use the selector to view sentiment/predictions, media, source coverage, model performance, human validation and events. Completed-month default: September 2026. October's partial collection is not treated as a complete month. Measures represent content supply, not respondent exposure. Factual calibration and subgroup comparisons are unavailable unless separately supplied.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
ROOT=Path('/content/drive/MyDrive/NLP-FinalProject')
SNAPSHOT='2026-10-16'
COMPLETE_THROUGH='2026-09-01'
# Set a specific Module 04 output folder if several analyses should be retained.
ANALYSIS_FOLDER=None
STUDY=ROOT/'03_ai_measurement/headline-measurement-v1'
MEDIA_RUN=STUDY/'model_runs/4ff1cce4591eb134'
ECON_RESULTS=ROOT/f'01_economic_baseline/results/{SNAPSHOT}'
ECON_PROCESSED=ROOT/f'01_economic_baseline/processed/{SNAPSHOT}'
print('Data snapshot:',SNAPSHOT,'| completed months through:',COMPLETE_THROUGH)

In [ ]:
def read_optional(path):
    if not path.exists():return pd.DataFrame()
    try:return pd.read_csv(path)
    except pd.errors.EmptyDataError:return pd.DataFrame()

def load_dashboard():
    folders=[p.parent for p in (ROOT/'04_association_analysis').glob('*/forecast_performance.csv')]
    selected=Path(ANALYSIS_FOLDER) if ANALYSIS_FOLDER else (max(folders,key=lambda p:(p/'forecast_performance.csv').stat().st_mtime) if folders else None)
    d={'analysis_folder':str(selected) if selected else 'Unavailable'}
    paths={
        'baseline':ECON_RESULTS/'out_of_sample_predictions.csv',
        'survey':ECON_PROCESSED/'monthly_current_official.csv',
        'media':MEDIA_RUN/'media_monthly_for_section4.csv',
        'outlets':MEDIA_RUN/'monthly_media_by_outlet.csv',
        'readiness':MEDIA_RUN/'monthly_scoring_readiness.csv',
        'agreement':MEDIA_RUN/'human_agreement.csv',
        'validation':MEDIA_RUN/'heldout_summary_metrics.csv',
        'events':STUDY/'economic_event_register.csv',
        'claims':MEDIA_RUN/'claims_official_verification.csv'
    }
    if selected:
        paths.update(performance=selected/'forecast_performance.csv',forecasts=selected/'rolling_forecasts.csv',
            associations=selected/'media_associations_hac.csv',diagnostics=selected/'residual_diagnostics.csv',
            sensitivity=selected/'sensitivity_common_dates.csv')
    for name,path in paths.items():
        frame=read_optional(path)
        for col in ['month','observation_month','analysis_month']:
            if col in frame:
                frame[col]=pd.to_datetime(frame[col],errors='coerce')
                frame=frame.loc[frame[col].le(pd.Timestamp(COMPLETE_THROUGH))].copy()
        d[name]=frame
    for key in ['performance','forecasts','associations','diagnostics','sensitivity']:
        d.setdefault(key,pd.DataFrame())
    return d

def note(text):display(Markdown(text))

def draw_dashboard(panel,platform,start,end):
    d=load_dashboard()
    lo=pd.Timestamp(start);hi=pd.Timestamp(end)
    def window(frame,col='month'):
        return frame.loc[frame[col].between(lo,hi)].sort_values(col) if col in frame else frame
    note(f'**Snapshot:** {SNAPSHOT}. **Completed-month limit:** {COMPLETE_THROUGH}. '
         f'**Display window:** {start} to {end}.')
    if panel=='Sentiment and gap':
        b=d['baseline']
        if b.empty:note('Economic baseline outputs unavailable.');return
        b=window(b.loc[b.target.eq('sentiment')&b.model.eq('economic_core')],'observation_month')
        if b.empty:note('No forecasts in selected window.');return
        fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True)
        axes[0].plot(b.observation_month,b.actual,color='black',label='First-archived sentiment')
        axes[0].plot(b.observation_month,b.predicted,label='Economic-only OOS prediction')
        if {'pi_lower','pi_upper'}.issubset(b):
            axes[0].fill_between(b.observation_month,pd.to_numeric(b.pi_lower,errors='coerce'),pd.to_numeric(b.pi_upper,errors='coerce'),alpha=.2,label='Nominal 95% interval')
        axes[0].legend();axes[0].set_ylabel('Index points')
        axes[1].plot(b.observation_month,b.sentiment_gap);axes[1].axhline(0,color='black',lw=.8)
        axes[1].set_ylabel('Observed minus predicted');fig.tight_layout();plt.show()
        note('Negative gaps are model-dependent unexplained sentiment, not proof of bias. Nominal intervals can under-cover; consult performance.')
        survey=window(d['survey'],'observation_month')
        cols=[c for c in ['sentiment','current_conditions','expectations'] if c in survey]
        if cols and len(survey):
            survey.set_index('observation_month')[cols].plot(figsize=(12,3),title='Survey components: current official history (descriptive)');plt.show()
    elif panel=='Media measures':
        m=window(d['media'])
        if m.empty:note('Monthly media export unavailable.');return
        families={
            'Tone':['negative_share','positive_share','emotional_share'],
            'Framing':['price_levels_share','household_frame_share','future_frame_share'],
            'Salience':['layoffs_share','hiring_share','rising_prices_share','falling_inflation_share','growth_expansion_share','recession_risk_share'],
            'Political attribution':['blame_share']}
        selected=['news','x'] if platform=='Both' else [platform.lower()]
        for title,measures in families.items():
            columns=[p+'_'+measure for p in selected for measure in measures if p+'_'+measure in m]
            if columns:
                ax=m.set_index('month')[columns].plot(figsize=(12,3.5),title=title)
                ax.set_ylabel('Share');ax.set_ylim(0,1);ax.legend(fontsize=8,ncol=2);plt.show()
        note('News: equal averages across contributing outlets. X: pooled eligible sampled posts. Denominators exclude unclassifiable labels; Mixed tone is included as classifiable. Rates do not measure exposure.')
        note('**Factual calibration: not assessed.** Official-claim detection is not verification; no accuracy time series is displayed.')
    elif panel=='Coverage and missingness':
        m=window(d['media'])
        if not m.empty:
            for columns,title in [(['news_scoring_completion','x_scoring_completion'],'Scoring completion'),(['news_eligible_items','x_eligible_items'],'Eligible scored content'),(['news_eligible_sources'],'Contributing news outlets')]:
                columns=[c for c in columns if c in m]
                if columns:m.set_index('month')[columns].plot(figsize=(12,3),title=title);plt.show()
            display(m[['month']+[c for c in m if c.endswith(('scoring_completion','eligible_items'))]].tail(12))
        o=window(d['outlets'],'analysis_month')
        if len(o):
            matrix=o.pivot(index='source',columns='analysis_month',values='eligible_items')
            fig,ax=plt.subplots(figsize=(13,max(4,len(matrix)*.3)))
            image=ax.imshow(np.ma.masked_invalid(matrix.to_numpy(float)),aspect='auto',cmap='Blues')
            ax.set_yticks(range(len(matrix)));ax.set_yticklabels(matrix.index)
            tick=list(range(0,len(matrix.columns),max(1,len(matrix.columns)//10)))
            ax.set_xticks(tick);ax.set_xticklabels([matrix.columns[i].strftime('%Y-%m') for i in tick],rotation=45)
            ax.set_title('Eligible scored headlines by outlet/month; blanks = unavailable, not zero')
            fig.colorbar(image,ax=ax,label='Headlines');fig.tight_layout();plt.show()
        note('These are counts in the scoring sample, not total output by each publisher. Geographic-unclear content is excluded from main aggregates. Failed requests and source composition can affect comparability.')
    elif panel=='Model performance':
        note('Performance tables use the full saved common evaluation window; display-date filters do not refit models.')
        for key,title in [('performance','Out-of-sample accuracy and coverage'),('associations','Media associations with HAC uncertainty and Holm adjustment'),('diagnostics','Residual diagnostics'),('sensitivity','Sensitivity checks on common dates')]:
            note('**'+title+'**');display(d[key]) if len(d[key]) else note('Not available.')
        f=window(d['forecasts'])
        if len(f):
            chart=f.pivot(index='month',columns='model',values='predicted')
            chart=chart[[c for c in ['M1_economics','M2_plus_media','M3_plus_controls','last_available'] if c in chart]]
            ax=chart.plot(figsize=(12,4));f.drop_duplicates('month').set_index('month').actual.plot(ax=ax,color='black',lw=2,label='Observed');ax.legend(fontsize=8);plt.show()
    elif panel=='Human validation':
        a=d['agreement'];v=d['validation']
        if len(a) and 'n' in a and pd.to_numeric(a.n,errors='coerce').gt(0).any():
            note('**Human agreement**');display(a.loc[a.grouping.eq('all')] if 'grouping' in a else a)
        else:note('Human agreement pending: no nonempty completed-label results found.')
        if len(v):
            note('**Held-out AI performance**');display(v.loc[v.grouping.eq('overall')] if 'grouping' in v else v)
        else:note('Held-out AI accuracy pending.')
        note('The reported completed annotation sample is 70 items (50 news, 20 X), below the assignment requirement of 300. Retain independent originals and report actual evaluation denominators.')
    else:
        e=d['events']
        if len(e):display(e)
        else:note('No populated verified event register. Candidate-release plots do not establish matched events.')
        note("""**Scope and causal limits**
- No respondent exposure data or audience weighting. Likes/reposts do not establish exposure.
- Demographic and political-affiliation subgroup estimates unavailable: no suitable subgroup inputs supplied.
- Survey-method comparability has not been formally adjusted here. Document Michigan collection-method changes before interpreting breaks.
- Campaign timing alone does not identify an election effect. This snapshot ends before election day.

**Prospective updates:** refresh Modules 01–02; score only new items with frozen settings and explicit budget; refresh validation and Module 03 aggregates; rerun Module 04; then press Refresh here. Preserve vintages, cutoff, configuration hashes and prior results. Advance completed-month cutoff only after that month is complete. No automatic paid updates occur.""")
    note('Analysis folder: `'+d['analysis_folder']+'`')

In [ ]:
panel=widgets.Dropdown(options=['Sentiment and gap','Media measures','Coverage and missingness','Model performance','Human validation','Events and limitations'],description='Panel:',layout=widgets.Layout(width='420px'))
platform=widgets.Dropdown(options=['Both','News','X'],description='Media:')
start=widgets.Text(value='2019-10-01',description='From:')
end=widgets.Text(value=COMPLETE_THROUGH,description='Through:')
refresh=widgets.Button(description='Refresh saved outputs',button_style='primary')
output=widgets.Output()
def refresh_view(_=None):
    with output:
        clear_output(wait=True)
        try:draw_dashboard(panel.value,platform.value,start.value,end.value)
        except Exception as exc:print(type(exc).__name__+': '+str(exc))
refresh.on_click(refresh_view)
panel.observe(refresh_view,names='value')
platform.observe(refresh_view,names='value')
display(widgets.VBox([widgets.HBox([panel,platform]),widgets.HBox([start,end,refresh]),output]))
refresh_view()

## Sharing and reproducibility
Keep this notebook with the four upstream notebooks and document input locations. The dashboard updates from saved outputs; it is not a hosted public service. Export chosen plots/tables for the report, or share the notebook with access to the required data files. Change `ANALYSIS_FOLDER` to select a specific saved regression run rather than the newest one. A pending field stays pending; missing data are never converted to reassuring zeros.